In [1]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 14.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 13.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 13.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 14.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0
    Uninstalling python-dateutil-2.9.0:
      Successfully uninstalled python-dateutil-2.9.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled n

In [1]:
import os
import json
import boto3
import glob
from urllib.parse import unquote_plus
from pyspark.sql import SparkSession
import uuid
import sys
import psycopg2
import logging
import pandas as pd
import janitor
from datetime import datetime
from pyspark.sql import SparkSession
from pandas.api.types import infer_dtype
from pyspark.sql.functions import col
import tempfile
from pandas.api.types import infer_dtype

In [2]:
formatter = logging.Formatter('[%(asctime)s] %(levelname)s @ line %(lineno)d: %(message)s')
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(formatter)
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(handler)

In [3]:
logger = logging.getLogger()
logger.setLevel("INFO")
ENDPOINT=os.getenv('ENDPOINT')
PORT=int(os.getenv('PORT'))
USER=os.getenv('USER')
REGION=os.getenv('REGION')
DBNAME=os.getenv('DBNAME')
DESTINATION_BUCKETNAME=os.getenv('DESTINATION_BUCKETNAME')
client = boto3.client('rds')
s3_client = boto3.client('s3')
logger = logging.getLogger()
logger.setLevel("INFO")
s3_client = boto3.client('s3')

[2025-04-23 11:34:59,310] INFO @ line 1147: Found credentials in environment variables.


In [4]:
dt_string = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
AppName = "RDSJOB"

In [ ]:
os.environ['DBNAME'] = 'piping'
os.environ['DB_PASSWORD']='Asimo1721.'
os.environ['AWS_REGION'] = 'eu-north-1'

In [5]:
def create_spark_session():
    try:
        spark = SparkSession.builder \
            .appName("sparkjum") \
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
            .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                    "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
            .config("spark.driver.extraClassPath", "/var/task/jars/*") \
            .master("local[*]") \
            .getOrCreate()
        return spark
    except Exception as e:
        logger.error(f"Error creating Spark session: {str(e)}")
        raise

In [6]:
sparkjum=create_spark_session()

In [7]:
def check_environment():
    required_vars = ['ENDPOINT', 'PORT', 'USER', 'REGION', 'DBNAME']
    missing_vars = [var for var in required_vars if not os.getenv(var)]
    if missing_vars:
        raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

In [17]:
# spark = (SparkSession
#     .builder
#     .appName(f"{AppName}_{dt_string}")
#     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
#     .config("spark.sql.catalog.s3tablesbucket", "org.apache.iceberg.spark.SparkCatalog")
#     .config("spark.sql.catalog.s3tablesbucket.catalog-impl", "software.amazon.s3tables.iceberg.S3TablesCatalog")
#     # .config("spark.sql.catalog.s3tablesbucket.warehouse", s3table_arn)
#     .config('spark.hadoop.fs.s3.impl', "org.apache.hadoop.fs.s3a.S3AFileSystem")
#     .config("spark.sql.defaultCatalog", "s3tablesbucket")
#     .config("spark.hadoop.fs.s3a.connection.timeout", "1200000")
#     .config("spark.hadoop.fs.s3a.path.style.access", "true")
#     .config("spark.hadoop.fs.s3a.connection.maximum", "200")
#     .config("spark.hadoop.fs.s3a.fast.upload", "true")
#     .config("spark.hadoop.fs.s3a.readahead.range", "256K")
#     .config("spark.hadoop.fs.s3a.input.fadvise", "random")
#     .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.EnvironmentVariableCredentialsProvider")
#     .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
#     .config("spark.hadoop.fs.s3a.endpoint", "eu-north-1.amazonaws.com")
#     .config("com.amazonaws.services.s3.enableV4", "true")
#     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
#     .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") 
#     .getOrCreate())

# spark.sparkContext.setLogLevel("DEBUG")
# logger.info("Spark session initialized successfully")

[2025-04-22 18:39:29,922] INFO @ line 25: Spark session initialized successfully


In [8]:

# def check_environment():
#     required_vars = ['ENDPOINT', 'PORT', 'USER', 'REGION', 'DBNAME']
#     missing_vars = [var for var in required_vars if not os.getenv(var)]
#     if missing_vars:
#         raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

# def query():
#     check_environment()
#     token = client.generate_db_auth_token(
#         DBHostname=ENDPOINT,
#         Port=PORT,
#         DBUsername=USER,
#         Region=REGION
#     )

#     try:
#         conn = psycopg2.connect(
#             host=ENDPOINT,
#             port=PORT,
#             database=DBNAME,
#             user=USER,
#             password='***.',
#             sslmode='require'  # Enforce SSL
#         )
#         cur = conn.cursor()
#         cur.execute("""SELECT * FROM support_control_2""")
#         query_results = cur.fetchall()
#         logger.info(query_results)
#         return query_results
#     except psycopg2.OperationalError as e:
#         logger.error(f"Connection failed: {e}")
#         raise
#     except Exception as e:
#         logger.error(f"Query failed: {e}")
#         raise
#     finally:
#         if 'cur' in locals() and cur:
#             cur.close()
#         if 'conn' in locals() and conn:
#             conn.close()

In [9]:
print("Current directory:", os.getcwd())
print("Files and directories:", os.listdir())

Current directory: /home/jovyan
Files and directories: ['.profile', '.bashrc', '.bash_logout', '.local', 'output', '.ipython', 'requirements.txt', '.npm', '.ipynb_checkpoints', '.jupyter', 'RDS.ipynb', 'TEIGA_TMI_ADISSEO_CONTROL_AVANCE_SOPORTACION.csv', '.cache', '.conda', '.config', '.wget-hsts', 'work']


In [10]:
input_path='TEIGA_TMI_ADISSEO_CONTROL_AVANCE_SOPORTACION2.csv'
output_path='output/'

In [11]:
def clean_column_name(column):
    # Replace spaces and special characters with underscores
    # Remove newline characters
    cleaned = column.replace(' ', '_').replace('/', '_').replace('\\n', '_').replace('\n', '_')
    # Remove any other special characters
    cleaned = ''.join(e for e in cleaned if e.isalnum() or e == '_')
    # Ensure the column name doesn't start with a number
    if cleaned[0].isdigit():
        cleaned = 'col_' + cleaned
    return cleaned.lower()  # Convert to lowercase for consistency

In [120]:

from pyspark.sql.functions import col, when, isnan, lit,isnull, trim, regexp_replace

def load_to_database_spark(df_spark):
    try:

        df_sample = df_spark.sample(withReplacement=False, fraction=0.3, seed=42).toPandas()

        # Step 2: Infer column types with janitor
        inferred_types = {}
        for col_name in df_sample.columns:
            dtype = infer_dtype(df_sample[col_name], skipna=True)
            logger.info(f"Inferred type for column {col_name}: {dtype}")
            inferred_types[col_name] = dtype

        # Step 3: Optional - cast types in Spark DataFrame based on inferred types
        cast_mapping = {
            'string': 'string',
            'unicode': 'string',
            'mixed': 'string',
            'mixed-integer': 'string',
            'integer': 'int',
            'floating': 'double',
            'boolean': 'boolean',
            'datetime64': 'TIMESTAMP_NTZ'
        }

        for col_name, inferred in inferred_types.items():
            spark_type = cast_mapping.get(inferred, 'string')
            df_spark = df_spark.withColumn(col_name, col(col_name).cast(spark_type))

        logger.info(f"Columns after casting: {df_spark.dtypes}")

        # Step 4: Save Spark DataFrame as a single CSV file
        with tempfile.TemporaryDirectory() as tmp_dir:
            output_path = os.path.join(tmp_dir, "spark_output")


            for c, dtype in df_spark.dtypes:
                if dtype == 'string':
                    df_spark = df_spark.withColumn(c, trim(col(c)))
                    df_spark = df_spark.withColumn(c, regexp_replace(col(c), r'["\n\r]', ' '))
            df_spark.coalesce(1).write.csv(output_path, header=True, mode='overwrite')
            # Find the CSV part file
            part_file = next((f for f in os.listdir(output_path) if f.startswith("part-") and f.endswith(".csv")), None)
            if not part_file:
                raise Exception("CSV part file not found")

            csv_file_path = os.path.join(output_path, part_file)

            # Step 5: Connect to database
            conn = psycopg2.connect(
                host=ENDPOINT,
                port=PORT,
                database=DBNAME,
                user=USER,
                password='Asimo1721.',
                sslmode='require'
            )
            cur = conn.cursor()

            # Step 6: Create table if it doesn't exist
            check_table_query = """
            SELECT EXISTS (
                SELECT FROM information_schema.tables 
                WHERE table_schema= 'public'
                AND table_name = 'source_support'
            );
            """
            cur.execute(check_table_query)
            table_exist = cur.fetchone()[0]

            if not table_exist:
                create_table_query = """
                    CREATE TABLE public.source_support (
                        id SERIAL PRIMARY KEY,
                        {}
                    )
                """.format(
                    ', '.join([f"{col} TEXT" for col in df_spark.columns])
                )
                cur.execute(create_table_query)
                conn.commit()

            # Step 7: Load data via COPY
            with open(csv_file_path, 'r', encoding='utf-8') as f:
                logger.info("Using COPY to load data into source_support...")
                cur.copy_expert(f"""
                    COPY public.source_support({', '.join(df_spark.columns)})
                    FROM STDIN WITH CSV HEADER
                """, f)

            conn.commit()
            logger.info("Data successfully loaded with COPY")

    except Exception as e:
        logger.error(f"Database operation failed: {e}")
        raise

    finally:
        if 'cur' in locals() and cur:
            cur.close()
        if 'conn' in locals() and conn:
            conn.close()


In [12]:
import tempfile
import os
from pyspark.sql.functions import col, when, isnan, lit, isnull, trim, regexp_replace
import janitor
from pandas.api.types import infer_dtype
import psycopg2
from typing import Dict, Optional
import logging

# Configure logging
logger = logging.getLogger(__name__)

def get_db_connection_params() -> Dict[str, str]:
    """Get database connection parameters from environment variables."""
    required_params = ['ENDPOINT', 'PORT', 'DBNAME', 'USER', 'DB_PASSWORD']
    
    params = {}
    for param in required_params:
        value = os.environ.get(param)
        if not value:
            raise ValueError(f"Missing required environment variable: {param}")
        params[param] = value
    
    return params

def load_to_database_spark(df_spark):
    conn = None
    cur = None
    
    try:
        # Step 1: Sample and infer types
        df_sample = df_spark.sample(withReplacement=False, fraction=0.3, seed=42).toPandas()
        
        # Step 2: Infer column types
        inferred_types = {
            col_name: infer_dtype(df_sample[col_name], skipna=True)
            for col_name in df_sample.columns
        }
        
        for col_name, dtype in inferred_types.items():
            logger.info(f"Inferred type for column {col_name}: {dtype}")

        # Step 3: Cast types in Spark DataFrame
        cast_mapping = {
            'string': 'string',
            'unicode': 'string',
            'mixed': 'string',
            'mixed-integer': 'string',
            'integer': 'int',
            'floating': 'double',
            'boolean': 'boolean',
            'datetime64': 'TIMESTAMP_NTZ'
        }
        logger.info(f"Row count before write: {df_spark.count()}")

        for col_name, inferred in inferred_types.items():
            spark_type = cast_mapping.get(inferred, 'string')
            df_spark = df_spark.withColumn(col_name, col(col_name).cast(spark_type))

        logger.info(f"Columns after casting: {df_spark.dtypes}")

        # Step 4: Clean and save data
        with tempfile.TemporaryDirectory() as tmp_dir:
            output_path = os.path.join(tmp_dir, "spark_output")

            # Clean string columns
            for c, dtype in df_spark.dtypes:
                if dtype == 'string':
                    df_spark = df_spark.withColumn(
                        c, 
                        regexp_replace(trim(col(c)), r'["\n\r]', ' ')
                    )

            df_spark.coalesce(1).write.csv(output_path, header=True, mode='overwrite')
            
            # Find CSV part file
            part_file = next(
                (f for f in os.listdir(output_path) 
                 if f.startswith("part-") and f.endswith(".csv")),
                None
            )
            
            if not part_file:
                raise FileNotFoundError("CSV part file not found")

            csv_file_path = os.path.join(output_path, part_file)

            # Step 5: Connect to database using environment variables
            db_params = get_db_connection_params()
            conn = psycopg2.connect(
                host=db_params['ENDPOINT'],
                port=db_params['PORT'],
                database=db_params['DBNAME'],
                user=db_params['USER'],
                password=db_params['DB_PASSWORD'],
                sslmode='require'
            )
            cur = conn.cursor()

            # Step 6: Create table if doesn't exist
            cur.execute("""
                SELECT EXISTS (
                    SELECT FROM information_schema.tables 
                    WHERE table_schema = 'public'
                    AND table_name = 'source_support'
                );
            """)
            
            if not cur.fetchone()[0]:
                columns_def = ', '.join(f"{col} TEXT" for col in df_spark.columns)
                create_table_query = f"""
                    CREATE TABLE public.source_support (
                        id SERIAL PRIMARY KEY,
                        {columns_def}
                    )
                """
                cur.execute(create_table_query)
                conn.commit()

            # Step 7: Load data via COPY
            with open(csv_file_path, 'r', encoding='utf-8') as f:
                logger.info("Loading data into source_support...")
                columns = ', '.join(df_spark.columns)
                copy_sql = f"""
                    COPY public.source_support({columns})
                    FROM STDIN WITH CSV HEADER
                """
                cur.copy_expert(copy_sql, f)

            conn.commit()
            logger.info("Data successfully loaded")

    except Exception as e:
        if conn:
            conn.rollback()
        logger.error(f"Database operation failed: {str(e)}")
        raise

    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()


In [32]:
# from pyspark.sql.functions import col
# from pandas.api.types import infer_dtype

# def load_to_database_spark(df_spark):
#     try:
#         # Clean column names
#         column_mapping = {col_name: clean_column_name(col_name) for col_name in df_spark.columns}
#         df_spark = df_spark.toDF(*[column_mapping[c] for c in df_spark.columns])

#         # Optional: show inferred types per column (Spark -> Pandas for display)
#         df_pandas_preview = df_spark.limit(100).toPandas()
#         for col_name in df_pandas_preview.columns:
#             logger.info(f"{col_name}: {infer_dtype(df_pandas_preview[col_name])}")

#         logger.info(f"CSV columns cleaned: {df_spark.columns}")

#         # Convert full Spark DataFrame to Pandas for DB load (if data size is small)
#         df_clean = df_spark.toPandas()

#         # Handle nulls
#         df_clean = df_clean.where(pd.notnull(df_clean), None)

#         # Connect to DB
#         conn = psycopg2.connect(
#             host=ENDPOINT,
#             port=PORT,
#             database=DBNAME,
#             user=USER,
#             password='Asimo1721.',
#             sslmode='require'
#         )
#         cur = conn.cursor()

#         # Check if table exists
#         check_table_query = """
#         SELECT EXISTS (
#             SELECT FROM information_schema.tables 
#             WHERE table_schema= 'public'
#             AND table_name = 'source_support'
#         );
#         """
#         cur.execute(check_table_query)
#         table_exist = cur.fetchone()[0]

#         if not table_exist:
#             create_table_query = """
#                 CREATE TABLE public.source_support (
#                     id SERIAL PRIMARY KEY,
#                     {}
#                 )
#             """.format(
#                 ', '.join([f"{col} TEXT" for col in df_clean.columns])
#             )
#             cur.execute(create_table_query)
#             conn.commit()

#         logger.info("Inserting data into source_support")
#         columns = ', '.join(df_clean.columns)
#         values = ','.join(['%s' for _ in df_clean.columns])
#         insert_query = f"""
#             INSERT INTO public.source_support ({columns})
#             VALUES ({values})
#         """

#         records = df_clean.values.tolist()
#         cur.executemany(insert_query, records)
#         conn.commit()

#         logger.info("Data successfully loaded to database")

#     except Exception as e:
#         logger.error(f"Database operation failed: {e}")
#         raise
#     finally:
#         if 'cur' in locals() and cur:
#             cur.close()
#         if 'conn' in locals() and conn:
#             conn.close()


In [105]:
df_spark = sparkjum.read.csv(input_path, header=True, inferSchema=True)
help(df_spark.write.csv)

Help on method csv in module pyspark.sql.readwriter:

csv(path: str, mode: Optional[str] = None, compression: Optional[str] = None, sep: Optional[str] = None, quote: Optional[str] = None, escape: Optional[str] = None, header: Union[bool, str, NoneType] = None, nullValue: Optional[str] = None, escapeQuotes: Union[bool, str, NoneType] = None, quoteAll: Union[bool, str, NoneType] = None, dateFormat: Optional[str] = None, timestampFormat: Optional[str] = None, ignoreLeadingWhiteSpace: Union[bool, str, NoneType] = None, ignoreTrailingWhiteSpace: Union[bool, str, NoneType] = None, charToEscapeQuoteEscaping: Optional[str] = None, encoding: Optional[str] = None, emptyValue: Optional[str] = None, lineSep: Optional[str] = None) -> None method of pyspark.sql.readwriter.DataFrameWriter instance
    Saves the content of the :class:`DataFrame` in CSV format at the specified path.
    
    .. versionadded:: 2.0.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
 

In [20]:
def process_csv(input_path, output_path):

    try:


        df_spark = sparkjum.read.csv(input_path, header=True, inferSchema=True)
        df_spark = df_spark.na.drop(how="all")
        logger.info(f"Row count before FUCNTION load_to_database_spark: {df_spark.count()}")


        logger.info(f"CSV columns: {df_spark.columns}")
        load_to_database_spark(df_spark)
        # df = df.groupby('E3DID').agg({'MONTAJE': 'sum'})
        # df.to_csv(output_path, index=False)
      
        

        return True
    except Exception as e:
        logger.error(f"Error processing CSV: {str(e)}")
        return False

In [19]:
process_csv(input_path,output_path)

[2025-04-23 11:47:38,306] INFO @ line 8: Row count before FUCNTION load_to_database_spark: 9461
[2025-04-23 11:47:38,310] INFO @ line 11: CSV columns: ['NM_REV', 'ID_MONMEGA', 'TAG', 'AREA', 'E3DID', 'ISOID', 'CLASS', 'DIAM_ITEM', 'SUPPORTID', 'SUPPORTMARKALL', 'SUPPORTMARK_or_SIKLATYPE', 'DATA1', 'DATA2or_SIKLAPARTNUMBER', 'DATA3', 'DATA4', 'DATA5', 'DATA6', 'DATA7', 'DATA8', 'DATA9', 'DATA10', 'REMARKS', 'VIGASparaCARTELAS', 'OLD_SUPPORT_ID', 'FABRICANTE', 'ESTADO_DE_FABRICACION_NUEVO_FORMATO', 'No_ENTREGA_NUEVO_FORMATO', 'Cuenta', 'FECHA', 'MONTAJE', 'FECHA2', 'PESO_KG', 'Observación', 'CONTEO', 'Weld', 'IV', 'Fecha_IV', 'Result', 'PT_', 'Fecha_PT', 'Result_PT', 'TEST_PACK_ASOCIADO_', 'SUPPORT']


/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):


[2025-04-23 11:47:38,868] INFO @ line 41: Inferred type for column NM_REV: string
[2025-04-23 11:47:38,869] INFO @ line 41: Inferred type for column ID_MONMEGA: string
[2025-04-23 11:47:38,870] INFO @ line 41: Inferred type for column TAG: string
[2025-04-23 11:47:38,872] INFO @ line 41: Inferred type for column AREA: string
[2025-04-23 11:47:38,874] INFO @ line 41: Inferred type for column E3DID: string
[2025-04-23 11:47:38,875] INFO @ line 41: Inferred type for column ISOID: string
[2025-04-23 11:47:38,876] INFO @ line 41: Inferred type for column CLASS: string
[2025-04-23 11:47:38,878] INFO @ line 41: Inferred type for column DIAM_ITEM: string
[2025-04-23 11:47:38,878] INFO @ line 41: Inferred type for column SUPPORTID: string
[2025-04-23 11:47:38,879] INFO @ line 41: Inferred type for column SUPPORTMARKALL: string
[2025-04-23 11:47:38,880] INFO @ line 41: Inferred type for column SUPPORTMARK_or_SIKLATYPE: string
[2025-04-23 11:47:38,881] INFO @ line 41: Inferred type for column DAT

True

In [31]:
df = pd.read_csv(input_path,header=0,sep=',')
df_spark = sparkjum.createDataFrame(df)
logger.info(f"Row count df: {df.count()}")
logger.info(f"Row count df_spark: {df_spark.count()}")

/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/utils.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/usr/local/spark/python/pyspark/sql/pandas/conversion.py:485: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if should_localize and is_datetime64tz_dtype(s.dtype) and s.dt.tz is not None:
/usr/local/spark/python/pyspark/sql/pandas/conversion.py:485: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if should_localize and is_datetime6

[2025-04-23 15:31:55,628] INFO @ line 3: Row count df: NM_REV                                 9447
ID_MONMEGA                             6122
TAG                                    9447
AREA                                   9447
E3DID                                  9447
ISOID                                  9175
CLASS                                  9410
DIAM_ITEM                              9410
SUPPORTID                              9447
SUPPORTMARKALL                         7824
SUPPORTMARK_or_SIKLATYPE               8375
DATA1                                  7796
DATA2or_SIKLAPARTNUMBER                7091
DATA3                                  6759
DATA4                                  3672
DATA5                                  3400
DATA6                                  2478
DATA7                                  1529
DATA8                                   265
DATA9                                   246
DATA10                                    1
REMARKS              